In [4]:
import sys
sys.path.insert(0,"/Users/chenry/code/ModelSEEDpy")
sys.path.append("/Users/chenry/code/cobrakbase")
#sys.path.append("/Users/piehl/Desktop/S2022/ModelSEEDpy-main/modelseedpy")
#sys.path.insert(0,"/Users/piehl/Desktop/S2022/ModelSEEDpy")
#sys.path.append("/Users/piehl/Desktop/S2022/cobrakbase-cobra-model")
import os
#os.environ["HOME"] = "/Users/piehl/Desktop/S2022"
import json
import cobra
import cobrakbase
from modelseedpy import MSPackageManager, MSGrowthPhenotypes
import pandas as pd
from IPython.display import HTML
import pandas as pd
from IPython.display import HTML
from modelseedpy import MSGapfill

In [5]:
kbase_api = cobrakbase.KBaseAPI()

In [6]:
#kbase_api = cobrakbase.KBaseAPI()
#List available models
modellist = kbase_api.ws_client.list_objects({
    "ids":[119455],
    "type":"KBaseFBA.FBAModel"})

In [7]:
model = kbase_api.get_from_ws("562.55917.fbamodel",119455)

In [8]:
#Pulling a media object
media = kbase_api.get_from_ws("Carbon-D-Glucose",117872)
#Using KBase media to constrain exchange reactions in model
pkgmgr = MSPackageManager.get_pkg_mgr(model,1)
pkgmgr.getpkg("KBaseMediaPkg",1).build_package(media, default_uptake=0, default_excretion=1000)

In [9]:
kbpheno = kbase_api.get_object("ecoli_biolog", 93541)
biolog = MSGrowthPhenotypes.from_kbase_object(kbpheno,kbase_api)  

In [13]:
mdlws = 119455
models = kbase_api.list_objects(mdlws, object_type="KBaseFBA.FBAModel",include_metadata=True)
#Loading cached data
#data = {}
f = open('BiologHash.json')
data = json.load(f)
gmm = kbase_api.get_from_ws("Carbon-Pyruvic-Acid","KBaseMedia")
template = kbase_api.get_from_ws("GramNegModelTemplateV4","NewKBaseModelTemplates")
for item in models:
    model = kbase_api.get_from_ws(item[1],mdlws)
    if item[1] not in data:
        data[item[1]] = {}
        model.objective = "bio1"
        msgapfill = MSGapfill(model,[template],[],{},{},[])
        gfresults = msgapfill.run_gapfilling(gmm,"bio1")
        model = msgapfill.integrate_gapfill_solution(gfresults)
        output = biolog.simulate_phenotypes(model,"bio1",True)
        for index, row in output["details"].iterrows():
            data[item[1]][row["Phenotype"]] = row["Simulated growth"]
        with open('BiologHash.json', 'w') as outfile:
            json.dump(data, outfile)

In [14]:
f = open('BiologHash.json')
data = json.load(f)
df = pd.DataFrame(data)
df.to_csv("BiologMacTestOutput.csv")